# SmokeSense ML-01: Baseline Model Development

**Project:** SmokeSense — Wildfire Smoke Prediction  
**Task:** ML-01 — Baseline A and Baseline B  
**Dataset:** smokesense_v1 (Version 1.0.0)  
**Author:** Miaoqing Wang

## Objective

The objective of ML-01 is to develop and evaluate two baseline
models for predicting future PM2.5 concentrations:

- **Baseline A: Persistence Model** — Assumes the most recent
  observed PM2.5 concentration remains unchanged.
- **Baseline B: Weather + Lag Model** — Uses historical PM2.5
  observations and weather features to predict future concentrations
  using a simple regression or tree-based model.

Both models will be evaluated at forecast horizons of
1, 3, 6, 12, and 24 hours.

### Evaluation Metrics

**Regression:** MAE and RMSE.

**Exceedance Classification:** Precision, Recall, F1-score,
AUROC, and AUPRC.

The results will establish reference performance for future
models, including XGBoost and temporal neural networks.

## 1. Dataset Loading and Initial Inspection

### 1.1 Dataset Description

This experiment uses the SmokeSense v1.0.0 dataset, provided
through the DATA-04 GitHub Release.

The dataset contains 141,954 observations across 20 wildfire
events and includes 38 columns.

Each observation represents a forecast point at a specific
issue time and forecast horizon.

The target variable is `target_pm25`, representing the PM2.5
concentration (µg/m³) at the corresponding future target time.

### 1.2 Data Loading

The dataset is stored in Apache Parquet format and loaded
using Pandas.

The accompanying `manifest.json` provides dataset version,
event grouping, data quality statistics, and provenance
information.

### 1.3 Initial Data Inspection

We examine:

1. Dataset dimensions and column names.
2. Data types and missing values.
3. Statistical properties of the target variable.
4. Observation counts across forecast horizons.

This inspection helps identify preprocessing requirements
before feature engineering and model training.

In [ ]:
from pathlib import Path
import pandas as pd
import json

def find_repo_root():
    current = Path.cwd().resolve()

    for path in [current, *current.parents]:
        if (path / ".git").exists() and (path / "backend" / "ml").exists():
            return path

    raise FileNotFoundError("SmokeSense repository not found.")

repo = find_repo_root()

dataset_dir = (
    repo / "backend" / "ml" / "data"
    / "processed" / "smokesense_v1"
)

with open(dataset_dir / "manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

df = pd.read_parquet(dataset_dir / "data.parquet")

print("Dataset:", manifest["name"])
print("Version:", manifest["version"])
print("Dataset shape:", df.shape)

display(df.head())


In [ ]:
# ML-01: Exploratory Data Analysis

# 1. Inspect data types
df.info()

# 2. Check missing values
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})

display(
    missing.sort_values(
        "Missing Percentage",
        ascending=False
    )
)

# 3. Inspect the target variable
display(df["target_pm25"].describe())

# 4. Count rows by forecast horizon
display(
    df["horizon_hours"].value_counts().sort_index()
)

## 2. Exploratory Data Analysis: Target Distribution

We examine the distribution of PM2.5 concentrations to understand
the prediction target, identify extreme smoke events, and guide
model evaluation.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(
    df["target_pm25"],
    bins=100,
    edgecolor="black"
)

plt.title("Distribution of Target PM2.5 Concentration")
plt.xlabel("PM2.5 (µg/m³)")
plt.ylabel("Number of Observations")

plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    df.loc[df["target_pm25"] <= 100, "target_pm25"],
    bins=50,
    edgecolor="black"
)

plt.title("PM2.5 Distribution (0–100 µg/m³)")
plt.xlabel("PM2.5 (µg/m³)")
plt.ylabel("Number of Observations")

### 2.1 Target Distribution Analysis

The PM2.5 target distribution is strongly right-skewed, with most observations concentrated at relatively low concentrations and a long tail extending to 798 µg/m³.

The mean concentration (23.21 µg/m³) is substantially higher than the median (10 µg/m³), confirming the influence of high-concentration observations.

**Implications for model development:**

- **Model evaluation:** Both MAE and RMSE will be reported. RMSE penalizes large prediction errors more heavily, making it useful for identifying models that perform poorly during extreme smoke events.
- **Exceedance detection:** Precision, recall, F1-score, AUROC, and AUPRC will be evaluated to assess whether the models can identify elevated PM2.5 concentrations.
- **Feature engineering:** Historical PM2.5 observations and weather conditions may help models capture changes associated with smoke events.
- **Model comparison:** Baseline B must demonstrate improvement over persistence on held-out data, particularly at the 6-hour forecast horizon.

No extreme PM2.5 observations will be removed solely because they are large. Such observations may represent important wildfire conditions, although their validity should be assessed separately.

In [ ]:
cols = [
    "issue_time",
    "horizon_hours",
    "pm25_lag_1h",
    "pm25_lag_3h",
    "pm25_lag_6h",
    "target_pm25",
]

display(df[cols].head(15))

print("Missing 1-hour lag:",
      df["pm25_lag_1h"].isna().sum())

## 3. Preprocessing and Feature Engineering

### 3.1 Handling Missing Historical PM2.5 Observations

The dataset contains 141,954 observations, of which 5,339 (3.76%) have missing `pm25_lag_1h` values.

For the initial persistence baseline, rows without a valid one-hour lag are excluded from evaluation because the baseline requires an available historical PM2.5 measurement.

The same eligible observations will be used when comparing Baseline A and Baseline B to ensure a fair comparison.

### 3.2 Leakage Prevention

The prediction target is `target_pm25`, representing the future PM2.5 concentration at `target_time`.

Only information available at or before the forecast issue time will be used as model input.

Related wildfire events must remain in the same training or evaluation group to reduce data leakage.

### 3.3 Feature Engineering Strategy

Baseline A uses an available historical PM2.5 observation directly and requires no parameter training.

Baseline B will combine PM2.5 lag observations with selected weather features. Missing values will be handled through a preprocessing pipeline fitted only on the training data.

The feature set and preprocessing transformations will be documented and reused during inference.

## 4. Baseline A: Persistence Model

### 4.1 Model Description

The persistence baseline assumes that future PM2.5 concentration remains equal to the most recent available historical observation.

For this initial experiment, `pm25_lag_1h` is used as the persistence input, pending confirmation of its exact timestamp definition.

The prediction equation is:


hat{y}_{t+h} = PM_{2.5}(t-1h)


where \(h\) represents the forecast horizon (1, 3, 6, 12, or 24 hours).

This model requires no training and serves as the reference that more sophisticated models must outperform.

### 4.2 Evaluation Metrics

The model will be evaluated using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE), separately for each forecast horizon.

The 6-hour horizon is the primary comparison target for SmokeSense.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Select observations eligible for persistence evaluation
baseline_a = df.dropna(
    subset=["pm25_lag_1h", "target_pm25"]
).copy()

# Persistence prediction
baseline_a["prediction"] = baseline_a["pm25_lag_1h"]

results = []

for horizon in [1, 3, 6, 12, 24]:
    subset = baseline_a[
        baseline_a["horizon_hours"] == horizon
    ]

    y_true = subset["target_pm25"]
    y_pred = subset["prediction"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    results.append({
        "Horizon (hours)": horizon,
        "Samples": len(subset),
        "MAE": mae,
        "RMSE": rmse
    })

baseline_a_results = pd.DataFrame(results)

display(baseline_a_results)

### 4.3 Baseline A Results and Interpretation

The persistence baseline was evaluated across five forecast horizons using observations with available `pm25_lag_1h` values.

The results demonstrate that prediction error increases as the forecast horizon becomes longer.

At the 1-hour horizon, the model achieved an MAE of 7.59 µg/m³ and an RMSE of 19.02 µg/m³. At the 24-hour horizon, MAE increased to 15.85 µg/m³ and RMSE increased to 37.37 µg/m³.

For the primary 6-hour forecasting target, the model achieved:

- **MAE:** 12.88 µg/m³
- **RMSE:** 30.61 µg/m³

RMSE was consistently greater than MAE, suggesting that some observations contribute large prediction errors.

These results provide an initial reference for Baseline B. However, they were computed across all eligible observations and should not be interpreted as final held-out test performance.

Before final model selection, we will evaluate both baselines using the same group-aware validation and test partitions.

In [ ]:
# Examine event groups before splitting

group_summary = (
    df.groupby("group_id")
    .agg(
        samples=("target_pm25", "size"),
        events=("pilot_event_id", "nunique"),
        start_time=("issue_time", "min"),
        end_time=("issue_time", "max")
    )
    .sort_values("start_time")
)

display(group_summary)

print("Number of groups:", df["group_id"].nunique())

### 3.4 Group-Aware Train/Validation/Test Split

The dataset contains 141,954 observations organized into 13 event groups representing 20 wildfire events.

The distribution of observations across groups is highly imbalanced. For example, `G-PE-001` accounts for approximately 44% of the dataset.

To reduce information leakage, we use a chronological group-based splitting strategy. Entire event groups are assigned to either training, validation, or testing, ensuring that related wildfire events do not appear in multiple partitions.

The partitions serve different purposes:

- **Training:** Fit model parameters and preprocessing transformations.
- **Validation:** Compare candidate models and select hyperparameters.
- **Testing:** Evaluate the final selected models on previously unseen wildfire event groups.

Because event groups vary substantially in size, the proportions of observations may differ from the intended proportions of groups.

The 6-hour forecast horizon is the primary evaluation target. Both baselines will be compared on identical eligible test observations.

In [ ]:
# Chronological group-aware train/validation/test split

group_order = (
    df.groupby("group_id")["issue_time"]
    .min()
    .sort_values()
    .index
    .tolist()
)

n_groups = len(group_order)

train_groups = group_order[:9]
val_groups = group_order[9:11]
test_groups = group_order[11:]

train_df = df[df["group_id"].isin(train_groups)].copy()
val_df = df[df["group_id"].isin(val_groups)].copy()
test_df = df[df["group_id"].isin(test_groups)].copy()

print("Training groups:", train_groups)
print("Validation groups:", val_groups)
print("Testing groups:", test_groups)

print("\nDataset sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# Verify no group overlap
assert set(train_groups).isdisjoint(val_groups)
assert set(train_groups).isdisjoint(test_groups)
assert set(val_groups).isdisjoint(test_groups)

print("\nGroup leakage check: PASSED")

In [ ]:
# Check temporal boundaries between splits

for name, subset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print("First issue:", subset["issue_time"].min())
    print("Last target:", subset["target_time"].max())

print("\nTrain/Validation temporal isolation:",
      train_df["target_time"].max() < val_df["issue_time"].min())

print("Validation/Test temporal isolation:",
      val_df["target_time"].max() < test_df["issue_time"].min())

### 3.5 Split Validation and Leakage Checks

The dataset was divided into training, validation, and testing partitions using chronological wildfire event groups.

The resulting partition sizes are:

| Partition | Observations | Percentage |
|---|---:|---:|
| Training | 104,710 | 73.8% |
| Validation | 2,760 | 1.9% |
| Testing | 34,484 | 24.3% |

The following checks were performed:

1. **Group isolation:** No `group_id` appears in more than one partition.
2. **Training/validation temporal isolation:** The latest training target timestamp precedes the earliest validation issue timestamp.
3. **Validation/test temporal isolation:** The latest validation target timestamp precedes the earliest test issue timestamp.

All three checks passed.

The resulting split provides chronological separation for model development and evaluation. However, feature-level leakage prevention must also be verified, and the small validation partition is a limitation when selecting models and hyperparameters.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

test_a = test_df.dropna(
    subset=["pm25_lag_1h", "target_pm25"]
).copy()

test_a["prediction"] = test_a["pm25_lag_1h"]

results = []

for horizon in [1, 3, 6, 12, 24]:
    subset = test_a[test_a["horizon_hours"] == horizon]

    results.append({
        "Horizon": horizon,
        "Samples": len(subset),
        "MAE": mean_absolute_error(
            subset["target_pm25"],
            subset["prediction"]
        ),
        "RMSE": np.sqrt(mean_squared_error(
            subset["target_pm25"],
            subset["prediction"]
        ))
    })

baseline_a_test_results = pd.DataFrame(results)

display(baseline_a_test_results)

### 4.4 Held-Out Test Evaluation

The persistence baseline was evaluated on two previously unseen event groups from 2024.

At the primary 6-hour forecast horizon, the model achieved an MAE of 5.79 µg/m³ and an RMSE of 17.78 µg/m³ across 6,193 eligible observations.

The MAE generally increased with forecast horizon, although the 24-hour MAE was slightly lower than the 12-hour MAE.

These test errors were lower than the corresponding exploratory errors across the full dataset. This may reflect differences in PM2.5 distributions and wildfire conditions between event groups.

The results establish a held-out reference for comparing Baseline B. The test set will not be used for feature selection or hyperparameter tuning.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

feature_columns = [
    "pm25_lag_1h",
    "pm25_lag_3h",
    "pm25_lag_6h",
    "pm25_lag_12h",
    "pm25_lag_24h",
    "temp_c",
    "rh_pct",
    "wind_speed_ms",
    "horizon_hours"
]

# Use comparable observations for both baselines
train_b = train_df.dropna(subset=["pm25_lag_1h"]).copy()
val_b = val_df.dropna(subset=["pm25_lag_1h"]).copy()

X_train = train_b[feature_columns]
y_train = train_b["target_pm25"]

X_val = val_b[feature_columns]
y_val = val_b["target_pm25"]

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Features:", feature_columns)

## 5. Baseline B: Weather and Lag Regression

### 5.1 Model Objective

Baseline B predicts future PM2.5 concentrations using historical PM2.5 measurements and observed weather conditions.

Unlike the persistence baseline, which assumes concentrations remain constant, Baseline B learns relationships between input features and future pollution concentrations.

We initially implement ordinary Linear Regression to establish a simple, interpretable machine learning baseline.

### 5.2 Feature Selection

The model uses:

- **Historical PM2.5:** 1, 3, 6, 12, and 24-hour lag measurements.
- **Weather:** Temperature, relative humidity, and wind speed.
- **Forecast horizon:** The number of hours between forecast issuance and the target time.

Fire-specific features are excluded initially to evaluate the predictive contribution of weather and historical pollution measurements.

### 5.3 Preprocessing

Missing feature values are replaced using median imputation, followed by standardization.

Both transformations are fitted exclusively on the training data to prevent information leakage.

A

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import numpy as np
import pandas as pd

horizons = [1, 3, 6, 12, 24]

# Each model is trained for a specific forecast horizon.
# horizon_hours is constant within each subset, so remove it.
model_features = [
    col for col in feature_columns
    if col != "horizon_hours"
]

baseline_b_models = {}
validation_results = []

for horizon in horizons:

    train_h = train_b[
        train_b["horizon_hours"] == horizon
    ]

    val_h = val_b[
        val_b["horizon_hours"] == horizon
    ]

    X_train_h = train_h[model_features]
    y_train_h = train_h["target_pm25"]

    X_val_h = val_h[model_features]
    y_val_h = val_h["target_pm25"]

    # Imputation and scaling are learned from training data only
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression())
    ])

    model.fit(X_train_h, y_train_h)

    predictions = np.maximum(model.predict(X_val_h), 0)

    # Persistence predictions on identical validation rows
    persistence_predictions = val_h["pm25_lag_1h"]

    validation_results.append({
        "Horizon": horizon,
        "Samples": len(val_h),
        "Persistence MAE": mean_absolute_error(
            y_val_h, persistence_predictions
        ),
        "Linear MAE": mean_absolute_error(
            y_val_h, predictions
        ),
        "Linear RMSE": np.sqrt(
            mean_squared_error(y_val_h, predictions)
        )
    })

    baseline_b_models[horizon] = model

validation_comparison = pd.DataFrame(validation_results)

display(validation_comparison.round(3))

### 5.5 Linear Regression Validation Results

Five independent Linear Regression models were trained using PM2.5 lag and weather features, one for each forecast horizon.

Validation results indicate that Linear Regression does not consistently outperform the persistence baseline.

At the primary 6-hour forecast horizon, Linear Regression achieved an MAE of 11.434 µg/m³ compared with 11.137 µg/m³ for Persistence, representing approximately 2.7% higher error.

At the 12-hour horizon, Linear Regression achieved an MAE of 11.360 µg/m³ compared with 12.205 µg/m³ for Persistence, demonstrating an improvement of approximately 6.9%.

Overall, these results suggest that a linear combination of historical PM2.5 and weather features does not consistently provide additional predictive value over persistence.

This motivates further experimentation with regularization and feature selection before selecting the final Baseline B model.

**Limitations:** The validation partition contains only two wildfire event groups. Results may be sensitive to event-specific conditions and should be interpreted cautiously.

In [ ]:
from sklearn.linear_model import Ridge

alphas = [0.1, 1, 10, 100, 1000]

ridge_results = []
ridge_models = {}

for horizon in [1, 3, 6, 12, 24]:

    train_h = train_b[train_b["horizon_hours"] == horizon]
    val_h = val_b[val_b["horizon_hours"] == horizon]

    X_train_h = train_h[model_features]
    y_train_h = train_h["target_pm25"]

    X_val_h = val_h[model_features]
    y_val_h = val_h["target_pm25"]

    for alpha in alphas:

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("regressor", Ridge(alpha=alpha))
        ])

        model.fit(X_train_h, y_train_h)

        predictions = np.maximum(model.predict(X_val_h), 0)

        mae = mean_absolute_error(y_val_h, predictions)

        ridge_results.append({
            "Horizon": horizon,
            "Alpha": alpha,
            "MAE": mae,
            "RMSE": np.sqrt(
                mean_squared_error(y_val_h, predictions)
            )
        })

        ridge_models[(horizon, alpha)] = model

ridge_results_df = pd.DataFrame(ridge_results)

# Select alpha independently for each horizon
best_ridge = (
    ridge_results_df
    .sort_values(["Horizon", "MAE", "Alpha"])
    .groupby("Horizon")
    .first()
    .reset_index()
)

display(best_ridge.round(3))

### 5.6 Ridge Regression Hyperparameter Tuning

Ridge Regression was evaluated as a regularized alternative to ordinary Linear Regression.

Five regularization strengths were tested: α = 0.1, 1, 10, 100, and 1000. Hyperparameters were selected independently for each forecast horizon using validation MAE.

At the primary 6-hour horizon, the best Ridge model used α = 1000 and achieved:

- **Validation MAE:** 11.425 µg/m³
- **Validation RMSE:** 25.854 µg/m³

This represents a small improvement over ordinary Linear Regression (MAE = 11.434), but does not outperform Persistence (MAE = 11.137).

At the 12-hour horizon, both Linear Regression and Ridge Regression achieved approximately 11.360 MAE, outperforming Persistence's 12.205 MAE.

Overall, Ridge regularization produced limited improvements. These findings suggest that additional feature engineering or models capable of representing nonlinear relationships may be necessary.

No conclusions about final test-set performance are drawn from these validation results.

In [ ]:
def add_engineered_features(frame):
    result = frame.copy()

    # Recent PM2.5 changes
    result["pm25_change_1_3"] = (
        result["pm25_lag_1h"] - result["pm25_lag_3h"]
    )

    result["pm25_change_3_6"] = (
        result["pm25_lag_3h"] - result["pm25_lag_6h"]
    )

    # Short-term PM2.5 average
    result["pm25_short_avg"] = result[
        ["pm25_lag_1h", "pm25_lag_3h", "pm25_lag_6h"]
    ].mean(axis=1)

    return result


train_fe = add_engineered_features(train_b)
val_fe = add_engineered_features(val_b)

engineered_features = model_features + [
    "pm25_change_1_3",
    "pm25_change_3_6",
    "pm25_short_avg"
]

print("Original features:", len(model_features))
print("Features after engineering:", len(engineered_features))

display(train_fe[engineered_features].head())

### 5.7 Feature Engineering

To investigate whether recent changes in PM2.5 improve prediction performance, three additional features were constructed from historical PM2.5 observations.

**Engineered features:**

1. `pm25_change_1_3`: Difference between the 1-hour and 3-hour PM2.5 lag measurements.
2. `pm25_change_3_6`: Difference between the 3-hour and 6-hour PM2.5 lag measurements.
3. `pm25_short_avg`: Mean of the 1-hour, 3-hour, and 6-hour PM2.5 lag measurements.

These features represent short-term pollution changes and recent concentration levels.

The engineered values are calculated using existing historical lag features without directly accessing future target concentrations.

Missing values are preserved for the difference features. The short-term average is calculated from available observations, so it may use fewer than three measurements when some values are missing.

The resulting dataset contains 11 model features, compared with 8 in the original regression experiment.

We will evaluate whether these transformations improve validation performance before deciding whether to retain them in Baseline B.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

fe_results = []
fe_models = {}

for horizon in [1, 3, 6, 12, 24]:

    train_h = train_fe[train_fe["horizon_hours"] == horizon]
    val_h = val_fe[val_fe["horizon_hours"] == horizon]

    # Reuse selected alpha from previous Ridge experiment
    alpha = float(
        best_ridge.loc[
            best_ridge["Horizon"] == horizon, "Alpha"
        ].iloc[0]
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=alpha))
    ])

    model.fit(
        train_h[engineered_features],
        train_h["target_pm25"]
    )

    predictions = np.maximum(
        model.predict(val_h[engineered_features]),
        0
    )

    fe_results.append({
        "Horizon": horizon,
        "Alpha": alpha,
        "Engineered MAE": mean_absolute_error(
            val_h["target_pm25"], predictions
        ),
        "Engineered RMSE": np.sqrt(
            mean_squared_error(
                val_h["target_pm25"], predictions
            )
        )
    })

    fe_models[horizon] = model

fe_results_df = pd.DataFrame(fe_results)

comparison = (
    best_ridge[["Horizon", "MAE"]]
    .rename(columns={"MAE": "Original Ridge MAE"})
    .merge(
        fe_results_df[["Horizon", "Engineered MAE", "Engineered RMSE"]],
        on="Horizon"
    )
)

comparison["MAE Improvement %"] = (
    (comparison["Original Ridge MAE"] - comparison["Engineered MAE"])
    / comparison["Original Ridge MAE"] * 100
)

display(comparison.round(3))

### 5.8 Feature Engineering Experiment Results

Three engineered features were added to the original weather-and-lag feature set: two PM2.5 change indicators and one short-term average.

The engineered Ridge models were evaluated using the same validation partition and regularization parameters selected in the previous experiment.

The results showed small reductions in MAE across most forecast horizons.

At the primary 6-hour horizon, MAE decreased from 11.425 to 11.394 µg/m³, representing an improvement of approximately 0.27%.

However, this result remained worse than the persistence baseline, which achieved an MAE of 11.137 µg/m³.

The limited improvement is consistent with the fact that the engineered variables are derived from existing linear combinations of PM2.5 lag measurements.

These results do not provide strong evidence that the additional features substantially improve prediction performance.

Further experiments with nonlinear models and evaluation across additional wildfire event groups may help determine their usefulness.

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

threshold = 35.5
classification_results = []

for horizon in [1, 3, 6, 12, 24]:
    val_h = val_fe[val_fe["horizon_hours"] == horizon]

    y_true = (val_h["target_pm25"] >= threshold).astype(int)

    model = fe_models[horizon]

    ridge_pred = np.maximum(
        model.predict(val_h[engineered_features]), 0
    )

    persistence_pred = val_h["pm25_lag_1h"].to_numpy()

    for name, predictions in [
        ("Persistence", persistence_pred),
        ("Engineered Ridge", ridge_pred)
    ]:
        predicted_events = (predictions >= threshold).astype(int)

        classification_results.append({
            "Horizon": horizon,
            "Model": name,
            "Precision": precision_score(
                y_true, predicted_events, zero_division=0
            ),
            "Recall": recall_score(
                y_true, predicted_events, zero_division=0
            ),
            "F1": f1_score(
                y_true, predicted_events, zero_division=0
            ),
            "AUROC": (
                roc_auc_score(y_true, predictions)
                if y_true.nunique() == 2 else np.nan
            ),
            "AUPRC": (
                average_precision_score(y_true, predictions)
                if y_true.nunique() == 2 else np.nan
            )
        })

classification_df = pd.DataFrame(classification_results)

display(classification_df.round(3))

In [ ]:
# Inspect validation class balance

threshold = 35.5

class_balance = []

for horizon in [1, 3, 6, 12, 24]:
    subset = val_b[val_b["horizon_hours"] == horizon]

    positives = (subset["target_pm25"] >= threshold).sum()
    total = len(subset)

    class_balance.append({
        "Horizon": horizon,
        "Total": total,
        "Positive Cases": positives,
        "Negative Cases": total - positives,
        "Positive Rate (%)": positives / total * 100
    })

class_balance_df = pd.DataFrame(class_balance)

display(class_balance_df.round(2))

### 5.9 PM2.5 Exceedance Classification Evaluation

To complement regression evaluation, both Persistence and Engineered Ridge were evaluated for their ability to identify elevated PM2.5 concentrations.

A provisional exceedance threshold of 35.5 µg/m³ was used, pending confirmation against the project's evaluation specification.

Precision, recall, F1-score, AUROC, and AUPRC were calculated for each forecast horizon.

At the primary 6-hour horizon:

| Metric | Persistence | Engineered Ridge |
|---|---:|---:|
| Precision | 0.143 | 0.120 |
| Recall | 0.143 | 0.143 |
| F1-score | 0.143 | 0.130 |
| AUROC | 0.676 | 0.637 |
| AUPRC | 0.108 | 0.093 |

Both models exhibited low recall, identifying approximately 14.3% of actual exceedance cases at the 6-hour horizon.

Persistence achieved higher F1, AUROC, and AUPRC than Engineered Ridge at the primary forecast horizon.

AUROC and AUPRC were calculated using continuous predicted PM2.5 concentrations as ranking scores rather than calibrated exceedance probabilities.

These results indicate that the current models have limited ability to detect elevated PM2.5 concentrations at the selected classification threshold.

Interpretation must account for the small validation dataset, class imbalance, and the provisional exceedance threshold.

### 5.10 Class Imbalance Analysis

The validation dataset exhibits substantial class imbalance when PM2.5 exceedance is defined using the provisional threshold of 35.5 µg/m³.

At the primary 6-hour forecast horizon, only 21 of 550 observations (3.82%) are classified as positive exceedance events.

Consequently, a classifier predicting no exceedances would achieve 96.18% accuracy while having zero recall for elevated PM2.5 events.

This illustrates why accuracy alone is inadequate for evaluating exceedance detection.

Precision, recall, F1-score, AUROC, and AUPRC provide more informative perspectives on performance under class imbalance.

The small number of positive cases also limits the reliability of validation estimates. A small change in correctly detected events can produce a substantial change in recall.

Future experiments should evaluate performance across additional wildfire event groups and consider alternative decision thresholds where appropriate. All threshold selection must occur using training or validation data, without using the held-out test set.

In [ ]:
### 5.11 Decision Tree Regression

To investigate whether nonlinear relationships improve forecasting performance, we evaluate a Decision Tree Regressor as an additional Baseline B candidate.

Unlike Linear Regression, Decision Trees partition the feature space into regions and can model nonlinear relationships and feature interactions.

We use the same weather and historical PM2.5 features, including the engineered features developed in Section 5.7.

To control model complexity, we experiment with two hyperparameters:

- `max_depth`: Maximum depth of the tree.
- `min_samples_leaf`: Minimum number of training samples required in a leaf node.

Hyperparameters are selected using validation MAE, with the 6-hour forecast as the primary project target.

Preprocessing and training are performed without using the held-out test dataset.

The objective is to determine whether a simple nonlinear model provides measurable improvements over Persistence and the existing linear regression baselines.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Hyperparameter candidates
max_depths = [3, 5, 8, 12]
min_leaf_sizes = [10, 30, 50, 100]

tree_results = []
tree_models = {}

for horizon in [1, 3, 6, 12, 24]:

    train_h = train_fe[
        train_fe["horizon_hours"] == horizon
    ]

    val_h = val_fe[
        val_fe["horizon_hours"] == horizon
    ]

    X_train_h = train_h[engineered_features]
    y_train_h = train_h["target_pm25"]

    X_val_h = val_h[engineered_features]
    y_val_h = val_h["target_pm25"]

    for depth in max_depths:
        for leaf_size in min_leaf_sizes:

            model = Pipeline([
                ("imputer", SimpleImputer(
                    strategy="median"
                )),
                ("regressor", DecisionTreeRegressor(
                    max_depth=depth,
                    min_samples_leaf=leaf_size,
                    random_state=42
                ))
            ])

            model.fit(X_train_h, y_train_h)

            predictions = np.maximum(
                model.predict(X_val_h), 0
            )

            mae = mean_absolute_error(
                y_val_h, predictions
            )

            rmse = np.sqrt(
                mean_squared_error(y_val_h, predictions)
            )

            tree_results.append({
                "Horizon": horizon,
                "Max Depth": depth,
                "Min Leaf": leaf_size,
                "MAE": mae,
                "RMSE": rmse
            })

            tree_models[(horizon, depth, leaf_size)] = model

tree_results_df = pd.DataFrame(tree_results)

best_tree = (
    tree_results_df
    .sort_values([
        "Horizon", "MAE", "Max Depth", "Min Leaf"
    ])
    .groupby("Horizon", as_index=False)
    .first()
)

display(best_tree.round(3))

In [ ]:
# Compare all models using validation MAE

model_comparison = (
    validation_comparison[
        ["Horizon", "Persistence MAE", "Linear MAE"]
    ]
    .merge(
        best_ridge[["Horizon", "MAE"]]
        .rename(columns={"MAE": "Ridge MAE"}),
        on="Horizon"
    )
    .merge(
        fe_results_df[["Horizon", "Engineered MAE"]],
        on="Horizon"
    )
    .merge(
        best_tree[["Horizon", "MAE"]]
        .rename(columns={"MAE": "Decision Tree MAE"}),
        on="Horizon"
    )
)

display(model_comparison.round(3))

In [ ]:
# Decision Tree classification evaluation
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

threshold = 35.5
tree_classification_results = []

for horizon in [1, 3, 6, 12, 24]:
    val_h = val_fe[val_fe["horizon_hours"] == horizon]

    selected = best_tree[
        best_tree["Horizon"] == horizon
    ].iloc[0]

    model = tree_models[(
        horizon,
        int(selected["Max Depth"]),
        int(selected["Min Leaf"])
    )]

    scores = np.maximum(
        model.predict(val_h[engineered_features]), 0
    )

    actual = (val_h["target_pm25"] >= threshold).astype(int)
    predicted = (scores >= threshold).astype(int)

    tree_classification_results.append({
        "Horizon": horizon,
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1": f1_score(actual, predicted, zero_division=0),
        "AUROC": roc_auc_score(actual, scores)
                 if actual.nunique() == 2 else np.nan,
        "AUPRC": average_precision_score(actual, scores)
                 if actual.nunique() == 2 else np.nan
    })

tree_classification_df = pd.DataFrame(tree_classification_results)
display(tree_classification_df.round(3))

### 5.13 Decision Tree Exceedance Evaluation

The selected Decision Tree models were evaluated for detecting PM2.5 concentrations exceeding the provisional threshold of 35.5 µg/m³.

At the primary 6-hour forecast horizon, Decision Tree achieved:

- Precision: 0.160
- Recall: 0.190
- F1-score: 0.174
- AUROC: 0.648
- AUPRC: 0.090

Compared with Persistence, Decision Tree improved precision, recall, and F1-score at this threshold. However, Persistence achieved higher AUROC and AUPRC.

The Decision Tree correctly identified approximately 4 of the 21 positive exceedance cases, indicating that most elevated PM2.5 events were still missed.

The results suggest that improved regression accuracy does not necessarily produce better exceedance ranking performance.

AUROC and AUPRC use predicted PM2.5 concentrations as continuous ranking scores, rather than calibrated exceedance probabilities.

The small number of positive validation cases limits the reliability of these estimates. Further evaluation across additional wildfire event groups is necessary.

In [ ]:
# Freeze selected Decision Tree configurations

selected_tree_config = (
    best_tree[
        ["Horizon", "Max Depth", "Min Leaf", "MAE", "RMSE"]
    ]
    .copy()
    .sort_values("Horizon")
    .reset_index(drop=True)
)

selected_tree_config["Model"] = "DecisionTreeRegressor"

display(selected_tree_config.round(3))

## 6. Final Held-Out Test Evaluation

### 6.1 Evaluation Strategy

The Decision Tree hyperparameters were selected using validation MAE without accessing the test dataset.

The selected models are now evaluated on held-out wildfire events from 2024.

For this evaluation, the models remain fitted on the original training partition. No additional hyperparameter tuning or feature selection is performed using test results.

Persistence and Decision Tree are evaluated on identical observations to ensure a fair comparison.

The evaluation reports MAE and RMSE for all five forecast horizons, with the 6-hour horizon serving as the primary project target.

The resulting test metrics will estimate performance on previously unseen event groups. Any limitations or performance degradation will be documented without changing the selected configurations based on test results.

In [ ]:
# Final held-out evaluation: Persistence vs Decision Tree

test_fe = add_engineered_features(
    test_df.dropna(subset=["pm25_lag_1h"]).copy()
)

final_results = []

for horizon in [1, 3, 6, 12, 24]:

    test_h = test_fe[
        test_fe["horizon_hours"] == horizon
    ]

    selected = selected_tree_config[
        selected_tree_config["Horizon"] == horizon
    ].iloc[0]

    model = tree_models[(
        horizon,
        int(selected["Max Depth"]),
        int(selected["Min Leaf"])
    )]

    y_true = test_h["target_pm25"]

    tree_pred = np.maximum(
        model.predict(test_h[engineered_features]), 0
    )

    persistence_pred = test_h["pm25_lag_1h"]

    persistence_mae = mean_absolute_error(
        y_true, persistence_pred
    )

    tree_mae = mean_absolute_error(
        y_true, tree_pred
    )

    final_results.append({
        "Horizon": horizon,
        "Samples": len(test_h),
        "Persistence MAE": persistence_mae,
        "Tree MAE": tree_mae,
        "Persistence RMSE": np.sqrt(
            mean_squared_error(y_true, persistence_pred)
        ),
        "Tree RMSE": np.sqrt(
            mean_squared_error(y_true, tree_pred)
        ),
        "MAE Improvement %": (
            (persistence_mae - tree_mae)
            / persistence_mae * 100
        )
    })

final_test_results = pd.DataFrame(final_results)

display(final_test_results.round(3))

### 6.2 Held-Out Test Results and Interpretation

The selected Decision Tree models were evaluated on previously unseen wildfire event groups from 2024.

Unlike the validation results, the Decision Tree did not outperform Persistence on test MAE at any of the five forecast horizons.

At the primary 6-hour horizon:

- Persistence MAE: 5.792 µg/m³
- Decision Tree MAE: 7.012 µg/m³
- Decision Tree RMSE: 16.890 µg/m³
- Persistence RMSE: 17.776 µg/m³

Decision Tree increased 6-hour MAE by approximately 21.1% compared with Persistence, despite reducing RMSE by approximately 5.0%.

At longer horizons, Decision Tree exhibited even larger increases in MAE, reaching approximately 65.1% at 24 hours.

These findings indicate that the improvements observed on the validation dataset did not generalize to the held-out test events under the primary MAE criterion.

Possible explanations include differences in wildfire conditions, event-specific feature distributions, and the limited size of the validation partition.

The experiment demonstrates the importance of evaluating machine learning models on independent data.

No additional hyperparameter tuning will be performed using the held-out test results.

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

threshold = 35.5
final_classification = []

for horizon in [1, 3, 6, 12, 24]:
    test_h = test_fe[test_fe["horizon_hours"] == horizon]

    config = selected_tree_config[
        selected_tree_config["Horizon"] == horizon
    ].iloc[0]

    model = tree_models[(
        horizon,
        int(config["Max Depth"]),
        int(config["Min Leaf"])
    )]

    y_true = (test_h["target_pm25"] >= threshold).astype(int)

    predictions = {
        "Persistence": test_h["pm25_lag_1h"].to_numpy(),
        "Decision Tree": np.maximum(
            model.predict(test_h[engineered_features]), 0
        )
    }

    for name, scores in predictions.items():
        y_pred = (scores >= threshold).astype(int)

        final_classification.append({
            "Horizon": horizon,
            "Model": name,
            "Positive Cases": int(y_true.sum()),
            "Precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_true, y_pred, zero_division=0
            ),
            "AUROC": (
                roc_auc_score(y_true, scores)
                if y_true.nunique() == 2 else np.nan
            ),
            "AUPRC": (
                average_precision_score(y_true, scores)
                if y_true.nunique() == 2 else np.nan
            )
        })

final_classification_df = pd.DataFrame(final_classification)
display(final_classification_df.round(3))

### 6.3 Held-Out Exceedance Classification Results

Both Persistence and the selected Decision Tree models were evaluated on the held-out 2024 wildfire event groups using a provisional PM2.5 exceedance threshold of 35.5 µg/m³.

At the primary 6-hour horizon, 135 of the 6,193 eligible observations were positive exceedance cases.

**Persistence achieved:**

- Precision: 0.298
- Recall: 0.289
- F1-score: 0.293
- AUROC: 0.784
- AUPRC: 0.237

**Decision Tree achieved:**

- Precision: 0.351
- Recall: 0.296
- F1-score: 0.321
- AUROC: 0.760
- AUPRC: 0.217

Decision Tree achieved slightly higher precision, recall, and F1-score at the fixed exceedance threshold. However, Persistence achieved better AUROC and AUPRC, suggesting better overall ranking of exceedance observations.

The results demonstrate that improvements in threshold-based classification performance do not necessarily correspond to improvements in concentration prediction accuracy or ranking performance.

At the primary 6-hour horizon, Persistence also achieved lower regression MAE than Decision Tree.

**Conclusion:** The selected Decision Tree did not establish a consistent improvement over Persistence on the held-out test dataset. Both models should be retained as experimental baselines, with Persistence serving as the reference benchmark.

The threshold remains provisional, and these results should not be interpreted as validated operational air-quality alerts.

## 7. Model Artifacts and Reproducibility

### 7.1 Objective

The purpose of this section is to save the trained Baseline B models and their associated metadata so that the experiments can be reproduced and the models can be reused by the SmokeSense forecasting system.

The selected Decision Tree models are stored separately for each forecast horizon: 1, 3, 6, 12, and 24 hours.

Each saved model includes the complete scikit-learn pipeline, including missing-value imputation and the trained Decision Tree Regressor.

The artifacts are initially stored in a local directory. They may later be published to GitHub Releases or approved cloud storage.

### 7.2 Model Selection

The Decision Tree was selected as the Baseline B candidate based on validation MAE. Its hyperparameters were fixed before evaluation on the held-out test set.

Although Decision Tree improved validation MAE at most horizons, Persistence achieved lower held-out test MAE at all five horizons.

Therefore, Persistence remains the reference benchmark, while Decision Tree is retained as the evaluated Baseline B candidate.

The test results are documented without using them to retune model hyperparameters.

### 7.3 Reproducibility

The saved artifacts include:

- Horizon-specific trained models and preprocessing pipelines.
- Selected hyperparameters.
- Model input feature names.
- Dataset version and SHA-256 checksum.
- Validation and test regression results.
- Exceedance classification evaluation results.

Feature engineering transformations must be applied consistently during both training and inference.

In [ ]:
import joblib
from pathlib import Path

artifact_dir = (
    repo / "backend" / "data" / "models"
    / "ml01_baselines"
)

models_dir = artifact_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

horizons = [1, 3, 6, 12, 24]

for horizon in horizons:

    config = selected_tree_config[
        selected_tree_config["Horizon"] == horizon
    ].iloc[0]

    depth = int(config["Max Depth"])
    leaf = int(config["Min Leaf"])

    model = tree_models[(horizon, depth, leaf)]

    model_path = models_dir / f"decision_tree_{horizon}h.joblib"

    joblib.dump(model, model_path)

    print(f"Saved: {model_path.name}")

print("\nAll five Decision Tree models saved.")

In [ ]:
import json
import sklearn
import pandas as pd

metadata = {
    "project": "SmokeSense",
    "task": "ML-01",
    "dataset_name": manifest["name"],
    "dataset_version": manifest["version"],
    "dataset_sha256": manifest["data_sha256"],
    "author": "Miaoqing Wang",
    "model_type": "DecisionTreeRegressor",
    "model_role": "Baseline B candidate",
    "training_partition": "train",
    "selection_partition": "validation",
    "evaluation_partition": "test",
    "forecast_horizons": horizons,
    "features": engineered_features,
    "engineered_feature_names": [
        "pm25_change_1_3",
        "pm25_change_3_6",
        "pm25_short_avg"
    ],
    "prediction_unit": "ug/m3",
    "exceedance_threshold": 35.5,
    "threshold_status": "provisional",
    "sklearn_version": sklearn.__version__,
    "selected_hyperparameters": {
        str(int(row["Horizon"])): {
            "max_depth": int(row["Max Depth"]),
            "min_samples_leaf": int(row["Min Leaf"])
        }
        for _, row in selected_tree_config.iterrows()
    }
}

with open(
    artifact_dir / "model_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=4)

model_comparison.to_csv(
    artifact_dir / "validation_results.csv",
    index=False
)

final_test_results.to_csv(
    artifact_dir / "test_results.csv",
    index=False
)

final_classification_df.to_csv(
    artifact_dir / "test_classification.csv",
    index=False
)

print("Metadata and evaluation results saved.")

In [ ]:
print("Saved artifacts:\n")

for path in sorted(artifact_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(artifact_dir))

## 8. Reusable Model Inference

### 8.1 Objective

The goal of this section is to convert the trained Baseline B models into a reusable prediction interface.

The inference workflow must reproduce the same feature engineering and preprocessing operations used during training.

For each prediction request, the system will:

1. Receive historical PM2.5 and weather features.
2. Generate the three engineered features.
3. Validate the required model input columns.
4. Load the trained model corresponding to the requested forecast horizon.
5. Predict future PM2.5 concentration.
6. Clip negative predictions to zero.

The saved scikit-learn pipelines include the fitted missing-value imputer and Decision Tree Regressor.

No model fitting or preprocessing parameter estimation occurs during inference.

### 8.2 Inference Consistency

Training and inference must use identical feature definitions, column ordering, and preprocessing transformations.

Prediction outputs are expressed in µg/m³ and are identified by forecast horizon.

The current implementation is an experimental ML-01 inference interface, not yet the production SmokeSense serving contract.

In [ ]:
import joblib
import numpy as np
import pandas as pd

from functools import lru_cache

@lru_cache(maxsize=5)
def load_baseline_model(horizon):
    if horizon not in (1, 3, 6, 12, 24):
        raise ValueError("Unsupported forecast horizon")

    model_path = (
        artifact_dir / "models"
        / f"decision_tree_{horizon}h.joblib"
    )

    return joblib.load(model_path)


def predict_baseline_b(frame, horizon):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("Input must be a Pandas DataFrame")

    # Verify that all raw input features exist
    missing = [
        col for col in model_features
        if col not in frame.columns
    ]

    if missing:
        raise ValueError(f"Missing input columns: {missing}")

    # Apply the same transformations as training
    processed = add_engineered_features(frame)

    # Load the selected model
    model = load_baseline_model(horizon)

    # Predict with the exact training feature order
    predictions = model.predict(
        processed[engineered_features]
    )

    return np.maximum(predictions, 0)

In [ ]:
# Select five 6-hour validation observations
sample = val_df[
    val_df["horizon_hours"] == 6
].head(5).copy()

predictions = predict_baseline_b(sample, horizon=6)

output = pd.DataFrame({
    "Actual PM2.5": sample["target_pm25"].to_numpy(),
    "Predicted PM2.5": predictions,
    "Persistence": sample["pm25_lag_1h"].to_numpy()
})

display(output.round(3))

In [ ]:
# Verify model serialization consistency

selected = selected_tree_config[
    selected_tree_config["Horizon"] == 6
].iloc[0]

original_model = tree_models[(
    6,
    int(selected["Max Depth"]),
    int(selected["Min Leaf"])
)]

processed_sample = add_engineered_features(sample)

original_predictions = np.maximum(
    original_model.predict(
        processed_sample[engineered_features]
    ),
    0
)

saved_predictions = predict_baseline_b(sample, 6)

np.testing.assert_allclose(
    original_predictions,
    saved_predictions,
    rtol=1e-10,
    atol=1e-10
)

print("Training/inference consistency: PASSED")

### 8.3 Inference Verification

The saved Decision Tree model was loaded using Joblib and tested on five validation observations at the 6-hour forecast horizon.

The inference function successfully generated nonnegative PM2.5 predictions using the same feature-engineering transformations applied during model development.

Repeated prediction values are expected because a Decision Tree Regressor assigns the same prediction to observations that reach the same terminal leaf.

A consistency test compares predictions generated by the original in-memory model and the serialized model to verify that model saving and loading preserve prediction behavior.

This test verifies the selected sample only; further integration tests will be needed before production deployment.

## 9. Standalone Inference Module

### 9.1 Objective

The objective of this section is to make the trained ML-01 baseline models accessible outside Jupyter Notebook.

The standalone inference module provides:

- Baseline A: Persistence forecasting.
- Baseline B: Decision Tree forecasting using saved model artifacts.
- Reusable feature engineering.
- Input validation and horizon-specific prediction.
- Nonnegative PM2.5 concentration predictions.

The module loads previously trained models without fitting them again.

### 9.2 Training and Inference Consistency

The inference module must use the same feature transformations and column ordering as the training notebook.

The fitted preprocessing pipelines are preserved in the saved Joblib artifacts.

The standalone interface is an experimental ML-01 component. Integration with the production model registry and SmokeSense serving contract will be addressed separately.

In [ ]:
from pathlib import Path

inference_dir = repo / "backend" / "ml" / "inference"
inference_dir.mkdir(parents=True, exist_ok=True)

module_path = inference_dir / "ml01_baselines.py"

print(module_path)

In [ ]:
import sys
import numpy as np

backend_dir = repo / "backend"

if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from ml.inference.ml01_baselines import predict

# Reuse our existing five validation observations
sample = val_df[
    val_df["horizon_hours"] == 6
].head(5).copy()

tree_predictions = predict(
    sample,
    horizon=6,
    model_type="tree",
    artifact_dir=artifact_dir,
)

persistence_predictions = predict(
    sample,
    horizon=6,
    model_type="persistence",
)

comparison = pd.DataFrame({
    "Actual": sample["target_pm25"].to_numpy(),
    "Decision Tree": tree_predictions,
    "Persistence": persistence_predictions,
})

display(comparison.round(3))

In [ ]:
#Verify the consistency
notebook_predictions = predict_baseline_b(sample, 6)

np.testing.assert_allclose(
    notebook_predictions,
    tree_predictions,
    rtol=1e-10,
    atol=1e-10,
)

print("Standalone inference consistency: PASSED")

## 10. Automated Testing and Integration

### 10.1 Objective

Automated tests are used to verify that the ML-01 inference module produces consistent predictions and handles invalid inputs correctly.

The tests cover:

- Persistence baseline predictions.
- Decision Tree predictions across five forecast horizons.
- Nonnegative and finite prediction outputs.
- Missing or invalid feature inputs.
- Unsupported forecast horizons.
- Consistency between notebook and standalone inference.

Testing is necessary to ensure that the trained models can be integrated into the SmokeSense forecasting pipeline reliably.

### 10.2 Testing Strategy

Unit tests focus on the behavior of individual functions, while artifact integration tests verify that the saved models can be loaded and used for inference.

The tests do not retrain models or modify the held-out test dataset.

The standalone inference module remains experimental until its feature definitions and interface are aligned with the project's serving contract.

In [ ]:
#Test the standalone module
import numpy as np
import pandas as pd

from ml.inference.ml01_baselines import predict

test_samples = val_df.head(10).copy()

for horizon in [1, 3, 6, 12, 24]:
    tree_pred = predict(
        test_samples,
        horizon=horizon,
        model_type="tree",
        artifact_dir=artifact_dir
    )

    assert len(tree_pred) == len(test_samples)
    assert np.isfinite(tree_pred).all()
    assert (tree_pred >= 0).all()

    print(f"{horizon}h: PASSED")

print("\nAll horizon tests passed.")

In [ ]:
# Test unsupported forecast horizon

try:
    predict(
        test_samples,
        horizon=7,
        model_type="tree",
        artifact_dir=artifact_dir
    )
    raise AssertionError("Invalid horizon was accepted")

except ValueError as error:
    print("Invalid horizon test: PASSED")
    print(error)


# Test missing required features

invalid_sample = test_samples.drop(
    columns=["pm25_lag_1h"]
)

try:
    predict(
        invalid_sample,
        horizon=6,
        model_type="tree",
        artifact_dir=artifact_dir
    )
    raise AssertionError("Missing feature was accepted")

except ValueError as error:
    print("\nMissing feature test: PASSED")
    print(error)

In [ ]:
#Verify Prediction consistency
import numpy as np
from ml.inference.ml01_baselines import predict

for horizon in [1, 3, 6, 12, 24]:

    sample_h = val_df[
        val_df["horizon_hours"] == horizon
    ].head(10).copy()

    standalone_pred = predict(
        sample_h,
        horizon=horizon,
        model_type="tree",
        artifact_dir=artifact_dir
    )

    notebook_pred = predict_baseline_b(
        sample_h,
        horizon
    )

    np.testing.assert_allclose(
        standalone_pred,
        notebook_pred,
        rtol=1e-10,
        atol=1e-10
    )

    print(f"{horizon}h consistency: PASSED")

## 11. Automated Testing with Pytest

### 11.1 Objective

Automated tests are developed to validate the behavior and reliability of the ML-01 inference module.

The tests verify:

- Persistence baseline predictions.
- Feature engineering calculations.
- Predictions for all five supported forecast horizons.
- Rejection of unsupported horizons and missing features.
- Consistency between saved model artifacts and direct pipeline predictions.

### 11.2 Testing Strategy

Pytest is used to execute repeatable unit and integration tests.

Unit tests verify functions using synthetic input data. Artifact integration tests verify that previously trained Decision Tree models can be loaded and used for forecasting.

The tests do not retrain models or use held-out test observations for hyperparameter selection.

The objective is to ensure the inference implementation is reproducible, reliable, and suitable for integration with the SmokeSense application.

In [ ]:
from pathlib import Path

tests_dir = repo / "backend" / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)

test_path = tests_dir / "test_ml01_baselines.py"

print(test_path)

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_ml01_baselines.py",
        "-v",
    ],
    cwd=repo / "backend",
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

In [ ]:
import subprocess

commands = [
    ["git", "branch", "--show-current"],
    ["git", "status", "--short"],
    ["git", "status", "--ignored", "--short",
     "backend/data/models/ml01_baselines"],
]

for command in commands:
    print("\n$", " ".join(command))

    result = subprocess.run(
        command,
        cwd=repo,
        capture_output=True,
        text=True
    )

    print(result.stdout or "(no output)")

    if result.returncode != 0:
        print(result.stderr)

In [ ]:
import subprocess

commands = [
    ["git", "fetch", "origin"],
    ["git", "log", "-1", "--oneline", "HEAD"],
    ["git", "log", "-1", "--oneline", "origin/dev"],
    ["git", "log", "--oneline", "origin/dev..HEAD"],
]

for command in commands:
    print("\n$", " ".join(command))

    result = subprocess.run(
        command,
        cwd=repo,
        capture_output=True,
        text=True
    )

    print(result.stdout or "(no output)")

    if result.returncode != 0:
        print("ERROR:", result.stderr)

In [ ]:
import subprocess

paths = [
    "backend/ml/notebooks/ML01_Baselines.ipynb",
    "backend/ml/inference/ml01_baselines.py",
    "backend/tests/test_ml01_baselines.py",
]

for path in paths:
    result = subprocess.run(
        ["git", "cat-file", "-e", f"origin/dev:{path}"],
        cwd=repo,
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print(f"EXISTS in origin/dev: {path}")
    else:
        print(f"NOT in origin/dev: {path}")